# TP bonus — La descente de gradient pas à pas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/ai_for_medicine/blob/main/ia-medecine/notebooks/iam-s3-bonus.ipynb)

*IA pour la médecine — séance 3, TP facultatif.*

**Objectifs**

1. comprendre la descente de gradient sur une fonction d'une variable ;
2. voir l'effet du **taux d'apprentissage** ;
3. ajuster une régression linéaire avec scikit-learn et retrouver les vrais paramètres.

C'est l'algorithme qui entraîne la régression logistique et les réseaux de
neurones : ici on le regarde sur l'exemple le plus simple possible.

## Préparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
# Fonctions d'affichage du TP : exécutez la cellule, inutile de la lire.
def plot_gradient_step(theta, theta_new, grad, learning_rate, f, theta_range):
    t = np.linspace(*theta_range, 200)
    plt.figure(figsize=(8, 4))
    plt.plot(t, f(t), "b-", lw=2, label="f(θ)")
    plt.plot(t, f(theta) + grad * (t - theta), "g--", alpha=0.6, label="tangente (pente = gradient)")
    plt.scatter([theta], [f(theta)], c="green", s=80, zorder=5, label=f"θ = {theta:.2f}")
    plt.scatter([theta_new], [f(theta_new)], c="red", s=80, zorder=5, label=f"θ nouveau = {theta_new:.2f}")
    plt.annotate("", xy=(theta_new, f(theta_new)), xytext=(theta, f(theta)),
                 arrowprops=dict(arrowstyle="->", color="red", lw=2))
    plt.ylim(min(f(t)) - 5, max(f(t)) * 1.05)
    plt.xlabel("θ"); plt.ylabel("f(θ)"); plt.legend(); plt.grid(alpha=0.3)
    plt.title(f"Un pas de descente de gradient (taux d'apprentissage = {learning_rate})")


def plot_gradient_descent_1d(f, theta_history, theta_range):
    t = np.linspace(*theta_range, 200)
    h = np.array(theta_history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(t, f(t), "b-", lw=2)
    axes[0].plot(h, f(h), "ro-", ms=4, alpha=0.7)
    axes[0].scatter([h[0]], [f(h[0])], c="green", s=80, zorder=5, label="départ")
    axes[0].scatter([h[-1]], [f(h[-1])], c="red", marker="*", s=200, zorder=5, label="fin")
    axes[0].set_xlabel("θ"); axes[0].set_ylabel("f(θ)"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[0].set_title("Trajectoire de θ")
    axes[1].plot(f(h), "o-")
    axes[1].set_xlabel("itération"); axes[1].set_ylabel("f(θ)"); axes[1].grid(alpha=0.3)
    axes[1].set_title("Valeur de la fonction à chaque itération")
    plt.tight_layout()


def plot_predictions(X, y, y_pred, title):
    order = np.argsort(X[:, 0])
    plt.figure(figsize=(8, 5))
    plt.scatter(X[:, 0], y, alpha=0.6, label="données")
    plt.plot(X[order, 0], y_pred[order], "r-", lw=2, label="prédiction")
    plt.xlabel("X"); plt.ylabel("y"); plt.title(title); plt.legend(); plt.grid(alpha=0.3)


def generate_linear_data(n_samples=100, n_features=1, noise=1.0, bias=5.0,
                         weight_scale=3.0, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3, 3, size=(n_samples, n_features))
    true_weights = rng.normal(0, weight_scale, size=n_features)
    y = X @ true_weights + bias + rng.normal(0, noise, size=n_samples)
    return X, y, true_weights, bias

---
# Partie 1 — Comprendre la descente de gradient

**But :** trouver la valeur de $\theta$ qui minimise une fonction $f(\theta)$.

La descente de gradient est itérative : à chaque pas, on se déplace dans le sens
opposé à la pente.

$$\theta_{\text{nouveau}} = \theta_{\text{ancien}} - \eta \cdot \nabla f(\theta_{\text{ancien}})$$

- $\theta$ : le paramètre à optimiser ;
- $\eta$ (êta) : le **taux d'apprentissage** (la taille du pas) ;
- $\nabla f(\theta)$ : le **gradient**, ici simplement la dérivée de $f$.

## Une fonction d'exemple

On minimise $f(\theta) = (3\theta - 7)^2$.

**Question :** où est le minimum de cette fonction ?

In [ ]:
def f(theta):
    """Fonction à minimiser : (3θ - 7)²"""
    return (3 * theta - 7) ** 2

def gradient_f(theta):
    """Dérivée de f : 2 × (3θ - 7) × 3 = 6 × (3θ - 7)"""
    return 6 * (3 * theta - 7)

theta_values = np.linspace(-2, 5, 100)
plt.figure(figsize=(8, 4))
plt.plot(theta_values, f(theta_values), "b-", linewidth=2)
plt.xlabel("θ"); plt.ylabel("f(θ)"); plt.title("f(θ) = (3θ - 7)²")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Minimum exact : θ = 7/3 ≈ {7/3:.4f}")

## Un pas de descente de gradient, en détail

In [ ]:
theta = 0.0
learning_rate = 0.05

grad = gradient_f(theta)
print(f"θ actuel       = {theta}")
print(f"f(θ) actuel    = {f(theta)}")
print(f"gradient en θ  = {grad}")
print(f"θ nouveau = θ - taux × gradient = {theta} - {learning_rate} × {grad} = {theta - learning_rate * grad}")

theta_new = theta - learning_rate * grad
plot_gradient_step(theta, theta_new, grad, learning_rate, f, theta_range=(-2, 5))
plt.show()

## La descente complète

On répète le pas un nombre fixé de fois :

In [ ]:
def gradient_descent(f, gradient_f, theta_init, learning_rate, n_iterations):
    """Descente de gradient ; renvoie la liste des θ successifs."""
    theta = theta_init
    theta_history = [theta]
    for i in range(n_iterations):
        grad = gradient_f(theta)
        theta = theta - learning_rate * grad
        theta_history.append(theta)
    return theta_history

theta_history = gradient_descent(f, gradient_f, theta_init=0.0,
                                 learning_rate=0.05, n_iterations=20)
print(f"θ de départ : {theta_history[0]:.4f}")
print(f"θ final     : {theta_history[-1]:.4f}")
print(f"vrai minimum : {7/3:.4f}")
plot_gradient_descent_1d(f, theta_history, theta_range=(-2, 5))
plt.show()

---
## Exercice 1 — Le taux d'apprentissage

Essayez plusieurs taux d'apprentissage : 0,01 (petit), 0,05 (moyen), 0,1 (grand),
0,12 (trop grand ?).

**Questions :**
1. Que se passe-t-il avec un très petit taux ?
2. Et avec un très grand taux ?
3. Quel taux convient à ce problème ?

In [ ]:
learning_rate = 0.05  # TODO : modifiez cette valeur (0.01, 0.05, 0.1, 0.12)

theta_history = gradient_descent(f, gradient_f, theta_init=0.0,
                                 learning_rate=learning_rate, n_iterations=20)
print(f"Taux : {learning_rate} → θ final : {theta_history[-1]:.4f} (cible {7/3:.4f})")
plot_gradient_descent_1d(f, theta_history, theta_range=(-2, 5))
plt.show()

*Votre réponse ici.*

---
## Exercice 2 — Une autre fonction

Minimisez $f_2(\theta) = \theta^2 + 5\theta + 6$. Écrivez la fonction, sa dérivée,
et le minimum exact (là où la dérivée s'annule), puis lancez la descente depuis
$\theta = 5$.

In [ ]:
def f2(theta):
    return ...  # TODO

def gradient_f2(theta):
    return ...  # TODO : la dérivée de f2

analytical_min = ...  # TODO : θ où la dérivée vaut 0

theta_history = gradient_descent(f2, gradient_f2, theta_init=5.0,
                                 learning_rate=0.1, n_iterations=30)
print(f"θ final : {theta_history[-1]:.4f} (minimum exact : {analytical_min})")
plot_gradient_descent_1d(f2, theta_history, theta_range=(-6, 6))
plt.show()

---
# Partie 2 — La régression linéaire avec scikit-learn

Problème d'apprentissage réel : trouver un poids $w$ et un biais $b$ tels que
$\hat{y} = Xw + b$ minimise l'erreur quadratique moyenne (MSE) :

$$\mathcal{L} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

## Un jeu de données simulé

On fabrique des données dont on connaît les vrais paramètres, pour vérifier que le
modèle les retrouve.

In [ ]:
X, y, true_weights, true_bias = generate_linear_data(
    n_samples=100, n_features=1, noise=1.0, bias=5.0, weight_scale=3.0, seed=42)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.6)
plt.xlabel("X"); plt.ylabel("y"); plt.title("Données simulées")
plt.grid(True, alpha=0.3)
plt.show()

## Ajuster le modèle

`LinearRegression` n'utilise pas la descente de gradient : pour la MSE, le minimum
a une formule exacte (moindres carrés ordinaires),

$$w = (X^T X)^{-1} X^T y$$

obtenue en une seule fois, sans taux d'apprentissage ni nombre d'époques. La
descente de gradient sert quand il n'existe pas de telle formule : régression
logistique, réseaux de neurones.

In [ ]:
model = LinearRegression()   # 1. créer : pas encore de paramètres
model.fit(X, y)              # 2. ajuster : calcule coef_ et intercept_
y_pred = model.predict(X)    # 3. prédire
mse = mean_squared_error(y, y_pred)  # 4. évaluer

print(f"Poids appris : {model.coef_} (vrai : {true_weights})")
print(f"Biais appris : {model.intercept_:.4f} (vrai : {true_bias})")
print(f"MSE : {mse:.4f}")
plot_predictions(X, y, y_pred, title="Prédictions du modèle")
plt.show()

---
## Exercice 3 — L'effet du bruit

Changez le niveau de bruit : 0,1 (données très propres), 1,0 (bruit modéré),
5,0 (très bruitées).

**Questions :**
1. Comment le bruit change-t-il la MSE ?
2. Les paramètres appris restent-ils proches des vrais ?

In [ ]:
noise_level = 1.0  # TODO : essayez 0.1, 1.0, 5.0

X_exp, y_exp, true_w, true_b = generate_linear_data(n_samples=100, noise=noise_level, seed=42)
model_exp = LinearRegression().fit(X_exp, y_exp)
y_pred_exp = model_exp.predict(X_exp)
print(f"Bruit : {noise_level} → MSE : {mean_squared_error(y_exp, y_pred_exp):.4f}")
print(f"Poids : {model_exp.coef_} (vrai : {true_w}), biais : {model_exp.intercept_:.3f} (vrai : {true_b})")
plot_predictions(X_exp, y_exp, y_pred_exp, title=f"Bruit = {noise_level}")
plt.show()

*Votre réponse ici.*

---
## Exercice 4 — Modifier le jeu de données

Essayez : plus d'exemples (`n_samples=500`), plus de bruit (`noise=3.0`), plus de
variables (`n_features=3`).

**Questions :**
1. Plus d'exemples change-t-il la MSE ? et la qualité des paramètres appris ?
2. Que se passe-t-il avec plus de variables ?

In [ ]:
X_new, y_new, true_w, true_b = generate_linear_data(
    n_samples=100,   # TODO : essayez 500
    n_features=1,    # TODO : essayez 3
    noise=1.0,       # TODO : essayez 3.0
    seed=42)
model_new = LinearRegression().fit(X_new, y_new)
y_pred_new = model_new.predict(X_new)
print(f"{X_new.shape[0]} exemples, {X_new.shape[1]} variables → MSE : {mean_squared_error(y_new, y_pred_new):.4f}")
print(f"Poids : {model_new.coef_} (vrais : {true_w}), biais : {model_new.intercept_:.3f}")

*Votre réponse ici.*